In [11]:
from scipy.signal import butter, lfilter
import ssvep_utils as su
import scipy.io as sio
from sklearn.cross_decomposition import CCA
from sklearn.metrics import confusion_matrix
import math
import numpy as np
import matplotlib.pyplot as plt

In [12]:
def bandpass_filter(data, lowcut, highcut, fs, order=5):
    nyq  = 0.5 * fs  # Nyquist Frequency
    low  = lowcut / nyq
    high = highcut /nyq
    b, a = butter(order, [low, high], btype='band')
    y    = lfilter(b, a, data)
    return y

In [13]:
def filter_set(data_set, lowcut, highcut, fs, order=5):
    filtered_set = np.zeros_like(data_set)
    for i in range(data_set.shape[0]):
        for j in range(data_set.shape[1]):
            filtered_set[i, j, :] = bandpass_filter(data_set[i, j, :], lowcut, highcut, fs, order=5)
    return filtered_set

In [14]:
def get_cca_reference_signals(data_len, target_freq, sampling_rate):
    reference_signals = []
    t = np.arange(0, (data_len/(sampling_rate)), step=1.0/(sampling_rate))
    reference_signals.append(np.sin(np.pi*2*target_freq*t))
    reference_signals.append(np.cos(np.pi*2*target_freq*t))
    reference_signals.append(np.sin(np.pi*4*target_freq*t))
    reference_signals.append(np.cos(np.pi*4*target_freq*t))
    reference_signals.append(np.sin(np.pi*6*target_freq*t))
    reference_signals.append(np.cos(np.pi*6*target_freq*t))
    reference_signals.append(np.sin(np.pi*8*target_freq*t)) # 4次谐波
    reference_signals.append(np.cos(np.pi*8*target_freq*t))
    reference_signals = np.array(reference_signals)
    
    return reference_signals

In [20]:
def find_correlation(n_components, np_buffer, freq):
    cca = CCA(n_components) # 指定提取的典型变量数
    corr = np.zeros(n_components)
    result = np.zeros(freq.shape[0])
    for freq_idx in range(0,freq.shape[0]):
        cca.fit(np_buffer.T,np.squeeze(freq[freq_idx, :, :]).T) # 拟合模型
        O1_a,O1_b = cca.transform(np_buffer.T, np.squeeze(freq[freq_idx, :, :]).T) # 转换为典型变量
        ind_val = 0
        for ind_val in range(0,n_components):
            corr[ind_val] = np.corrcoef(O1_a[: ,ind_val], O1_b[:, ind_val])[0 ,1]
            result[freq_idx] = np.max(corr)

    
    return result

In [21]:
def cca_classify(segmented_data, reference_templates):
    predicted_class = []
    labels = []
    # the struture of segmented data (targets, channels, trials, segments, samples)
    for target in range(0, segmented_data.shape[0]):
        for trial in range(0, segmented_data.shape[2]):
            for segment in range(0, segmented_data.shape[3]):
                labels.append(target)
                result = find_correlation(1, segmented_data[target, :, trial, segment, :], 
                                      reference_templates)
                predicted_class.append(np.argmax(result)+1)
    labels = np.array(labels)+1
    predicted_class = np.array(predicted_class)

In [22]:
data_path = r'D:\2024\7-9\dissertation\code'
all_segment_data = dict()
all_acc = list() # accuracy
window_len = 1
shift_len = window_len # no overlap data in segments

# Parameters for benchmark dataset
fs = 250
n_subjects = 10
n_freqs = 10
n_chans = 64
n_trials = 6
t_cue = 0.5

# Filtering
lowcut = 7
highcut = 90

# CCA classification
t_select = window_len # 1s
t_start = 2
n_start = t_start * fs
t_sum = t_cue + t_select

duration = int(window_len * fs)
flicker_freq = np.array([8.0, 8.2, 8.4, 8.6, 8.8, 
                         9.0, 9.2, 9.4, 9.6, 9.8]) #闪烁频率

In [23]:
#Load data
for subject in np.arange(1, 2):
#for subject in np.arange(1, n_subjects+1):
    dataset = sio.loadmat(f'{data_path}/S{subject}.mat') # channels, samples, targets, trials
    eeg = np.array(dataset['data'][:n_chans, n_start:, :n_freqs, :], dtype='float32')
    eeg = eeg.transpose(2, 0, 1, 3) # targets, channels, samples, trials

    filtered_data = filter_set(eeg, lowcut, highcut, fs, order=5) # Same as eeg & FB-eeg
    all_segment_data[f'S{subject}'] = su.get_segmented_epochs(filtered_data, window_len, 
                                shift_len, fs) # targets, channels, trials, segments, samples

# Generate sinusoidal templates 正弦波模板 for the given 10-class SSVEP datasets 
# (We know the freqs already)
reference_templates = []
for fr in range(0, len(flicker_freq)):
    reference_templates.append(get_cca_reference_signals(duration, flicker_freq[fr], sample_rate))
reference_templates = np.array(reference_templates, dtype='float32')

#Perform classification using cca_classify func
for subject in all_segment_data.keys():
    labels, predicted_class = cca_classify(all_segment_data[subject], reference_templates)
    c_mat = confusion_matrix(labels, predicted_class)
    accuracy = np.divide(np.trace(c_mat), np.sum(np.sum(c_mat)))
    all_acc.append(accuracy)
    print(f'Subject: {subject}, Accuracy: {accuracy*100} %')
#    print(f'Subject: {subject}, Confusion matrix: {c_mat} ')
print(f'avg acc:{np.mean(all_acc)*100}%')    

IndexError: too many indices for array: array is 2-dimensional, but 3 were indexed